# <b>Diffusion2ch -> PnL features: synthetic data sampling.</b>

## What this notebook does

This notebook creates intermediate split files with `diff2ch_*` features built on synthetic liquidation forecasts:

```text
plain split parquet files
    ↓
conditional 2ch Diffusion-TS forecast features
    ↓
split parquet files with diff2ch_* columns
    ↓
2_Results / walk_forward with advanced + diff2ch features
```

## How generation works

Generation is organized as follows:

1. **Per-window training.** For each walk-forward window (`W1`–`W5`), a separate 2-channel conditional Diffusion-TS model (`log_delta_t`, `log_dollar_amount`) is trained only on liquidation events from the `train_model` period. `CopulaSpaceTransform` and `MinMaxScaler` are also fitted only on train, so there is no leakage into `calib` / `test`.
2. **Forecast anchors.** Forecasts are generated not for every trade, but on a sparse grid of liquidation anchors: `FORECAST_ANCHOR_STRIDE_EVENTS = 64` (every 64th liquidation event), with a limit of `MAX_FORECAST_ANCHORS_PER_WINDOW = 500` anchors per window, spread uniformly across the whole split period.
3. **Conditional sampling.** For each anchor, a context of the last `HIST_LEN = 192` events is taken, and the model samples `N_SAMPLES = 16` synthetic future paths of `PRED_LEN = 64` events each (`FORECAST_SAMPLING_STEPS = 100`).
4. **Aggregation into features.** Synthetic paths are aggregated into `diff2ch_*` features: expected number of liquidations, expected dollar volume, p90/p95 of volume and the probability of at least one liquidation over 30 / 120 / 300 second horizons, plus characteristics of the time to the next liquidation.
5. **Asof-join to trades.** Features are joined to trade-level split data via backward `join_asof` on the latest available anchor. Additionally, `time_since_diff2ch_anchor` and `time_since_last_liq` are computed, so the downstream model can account for forecast staleness.

## ! Important note on coverage and compute

With the current parameters we get **suboptimal coverage**, and this is a deliberate trade-off in favor of speed:

- `N_SAMPLES = 16` is too few for a stable estimate of tail quantiles (p90/p95) and probabilities of rare events; according to the diagnostics from the forecasting notebook, even with K=32 the coverage of the 90% intervals is only ~79–81% instead of the target 90%;
- `FORECAST_ANCHOR_STRIDE_EVENTS = 64` and `MAX_FORECAST_ANCHORS_PER_WINDOW = 500` make the anchor grid sparse — a significant share of trades receives a stale forecast, which degrades the features during the asof-join;
- for production quality, **more compute** is needed: increase `N_SAMPLES` (64–128+), reduce the anchor stride (ideally down to 1) and remove the limit on the number of anchors. This is several times more expensive in GPU time, so the current configuration is used as a smoke-test / first-pass experiment.

In [ ]:
# Imports
import os
import sys
import gc
import copy
import importlib.util
from pathlib import Path

import yaml
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from scipy.stats import norm
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader


def import_module_with_fallback(module_name: str, candidate_files: list[str]):
    """Try ordinary import, then local .py file fallback."""
    try:
        return __import__(module_name)
    except ModuleNotFoundError:
        pass

    for filename in candidate_files:
        path = Path.cwd() / filename
        if path.exists():
            spec = importlib.util.spec_from_file_location(module_name, path)
            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)
            sys.modules[module_name] = module
            return module

    raise ModuleNotFoundError(
        f"Could not import {module_name}. Put one of {candidate_files} near this notebook."
    )


try:
    from meta import load_split_dataset
    from splits import get_window_ids
except Exception as e:
    print("Could not import project modules meta/splits.")
    print("Run this notebook from Experiment Pipeline project root.")
    raise e


ut = import_module_with_fallback(
    module_name="utilities_transforms",
    candidate_files=["utilities_transforms.py", "utilities_transforms(1).py"]
)


def find_repo_dir() -> Path:
    """Find local Diffusion-TS repository."""
    candidates = [
        Path.cwd(),
        Path.cwd() / "Diffusion-TS",
        Path.cwd().parent / "Diffusion-TS",
        Path(r"Diffusion-TS")
    ]

    for repo_dir in candidates:
        if (repo_dir / "engine" / "solver.py").exists():
            return repo_dir

    raise FileNotFoundError(
        "Diffusion-TS repo was not found. Add your repo path to candidates in find_repo_dir()."
    )


REPO_DIR = find_repo_dir()
sys.path.insert(0, str(REPO_DIR))

from engine.solver import Trainer
from Utils.io_utils import load_yaml_config, instantiate_from_config

print("REPO_DIR:", REPO_DIR)
print("CUDA available:", torch.cuda.is_available())


## <b>1. Global config.</b>

Check the paths before running. For the first smoke test, it is best to set:

```python
WINDOWS_TO_RUN = [‘W1’]
MAX_FORECAST_ANCHORS_PER_WINDOW = 500
N_SAMPLES = 16
```

Once you have checked everything, you can remove the limit and increase `N_SAMPLES`.


In [ ]:

# Main user config
INSTRUMENT = "btc"       # "btc" or "eth"
WINDOWS_TO_RUN = ["W1","W2","W3","W4","W5"]  # or ["W1"] for a quick smoke test

# Input split files without meta/diffusion features.
BASE_SPLIT_DIR = Path(f"data/meta/{INSTRUMENT}_splits_only")

# Output split files with diff2ch_* features.
OUT_SPLIT_DIR = Path(f"data/meta/{INSTRUMENT}_splits_diff2ch")
OUT_SPLIT_DIR.mkdir(parents=True, exist_ok=True)

# Raw liquidation-event data. Must contain: timestamp, price, amount.
LIQ_PATH = Path(f"data/binance_liquidations/perp_{INSTRUMENT}usdt.parquet")

# Diffusion-TS config. This notebook copies an existing config and patches seq/channel dims.
SOURCE_CONFIG_PATH = Path(
    r"Diffusion-TS/Config/liquidations.yaml"
)

EXP_ROOT = REPO_DIR / f"exp_diff2ch_to_pnl_{INSTRUMENT}"
EXP_ROOT.mkdir(parents=True, exist_ok=True)

# Conditional event-window structure.
HIST_LEN = 192
PRED_LEN = 64
SEQ_LEN = HIST_LEN + PRED_LEN
FEATURE_COLUMNS = ["log_delta_t", "log_dollar_amount"]
INPUT_CHANNELS = len(FEATURE_COLUMNS)

# Diffusion training.
BATCH_SIZE = 64
TRAIN_WINDOW_STRIDE = 64
RUN_TRAIN = True
LOAD_MILESTONE = None   # e.g. 10 to load checkpoint instead of training from scratch.

# Conditional sampling for feature generation.
N_SAMPLES = 16
SAMPLE_BATCH = 128
FORECAST_COEF = 1e-3
FORECAST_STEPSIZE = 1e-3
FORECAST_SAMPLING_STEPS = 100

# We generate forecasts on liquidation anchors and then asof-join to trades.
# stride=1 is best but expensive. Start with 16 or 64 for a quick experiment.
FORECAST_ANCHOR_STRIDE_EVENTS = 64
MAX_FORECAST_ANCHORS_PER_WINDOW = 500  # e.g. 1000 for smoke test; None for full run.

HORIZONS_S = [30, 120, 300]
OVERWRITE = True

print("INSTRUMENT:", INSTRUMENT)
print("WINDOWS_TO_RUN:", WINDOWS_TO_RUN)
print("BASE_SPLIT_DIR:", BASE_SPLIT_DIR)
print("OUT_SPLIT_DIR:", OUT_SPLIT_DIR)
print("LIQ_PATH:", LIQ_PATH)
print("SEQ_LEN:", SEQ_LEN, "=", HIST_LEN, "+", PRED_LEN)
print("N_SAMPLES:", N_SAMPLES)
print("FORECAST_ANCHOR_STRIDE_EVENTS:", FORECAST_ANCHOR_STRIDE_EVENTS)


## <b>2. Datasets, transforms and Diffusion-TS helpers</b>

In [ ]:
# Dataset classes
class NumpyWindowDataset(Dataset):
    """Minimal dataset compatible with Diffusion-TS Trainer."""
    def __init__(self, data: np.ndarray):
        assert data.ndim == 3
        self.data = torch.tensor(data, dtype=torch.float32)
        self.window = int(data.shape[1])
        self.var_num = int(data.shape[2])
        self.auto_norm = False

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        return self.data[idx]


class ConditionalWindowDataset(Dataset):
    """
    Dataset for mask-based conditional generation.

    mask == True  : known values, fixed during sampling.
    mask == False : unknown values, sampled by diffusion model.
    """
    def __init__(self, data: np.ndarray, mask: np.ndarray):
        assert data.shape == mask.shape
        assert data.ndim == 3
        self.data = torch.tensor(data, dtype=torch.float32)
        self.mask = torch.tensor(mask, dtype=torch.bool)
        self.window = int(data.shape[1])
        self.var_num = int(data.shape[2])
        self.auto_norm = False

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        return self.data[idx], self.mask[idx]


def make_windows(x: np.ndarray, seq_len: int, stride: int) -> np.ndarray:
    """Build time-ordered windows with a chosen stride."""
    x = np.asarray(x, dtype=np.float32)
    if x.ndim != 2:
        raise ValueError("x must have shape (n_events, n_channels).")
    if len(x) < seq_len:
        raise ValueError(f"Not enough events for one full window: {len(x)} < {seq_len}")

    starts = np.arange(0, len(x) - seq_len + 1, stride)
    return np.stack([x[s:s + seq_len] for s in starts], axis=0).astype(np.float32)


def make_forecast_mask(windows: np.ndarray, hist_len: int) -> np.ndarray:
    """Known prefix mask for conditional forecasting."""
    mask = np.zeros_like(windows, dtype=bool)
    mask[:, :hist_len, :] = True
    return mask


def to_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


In [ ]:

# Train-fitted empirical copula and MinMax transforms
def empirical_copula_transform_from_train(
    new_values: np.ndarray,
    train_values: np.ndarray,
    eps: float = 1e-10,
) -> np.ndarray:
    """
    Real space -> train-fitted empirical CDF -> normal copula space.
    Uses only train_values, so it is safe for calib/test.
    """
    new_values = np.asarray(new_values, dtype=np.float64).reshape(-1)
    train_sorted = np.sort(np.asarray(train_values, dtype=np.float64).reshape(-1))

    if train_sorted.size == 0:
        raise ValueError("train_values must be non-empty.")

    ranks = np.searchsorted(train_sorted, new_values, side="right")
    u = (ranks + 0.5) / (train_sorted.size + 1.0)
    u = np.clip(u, eps, 1.0 - eps)
    return norm.ppf(u)


def minmax_transform_from_train(
    copula_values: np.ndarray,
    mm_transform,
    clip: bool = True,
) -> np.ndarray:
    """Copula space -> train-fitted min-max model space."""
    denom = mm_transform._max - mm_transform._min
    if denom == 0:
        raise ValueError("MinMaxScaler has zero range.")

    out = (copula_values - mm_transform._min) / denom
    if clip:
        out = np.clip(out, 0.0, 1.0)
    return out


def fit_2ch_transforms(train_raw: pd.DataFrame) -> dict:
    """Fit CopulaSpaceTransform and MinMaxScaler on train events only."""
    cs_transforms = {}
    mm_transforms = {}
    train_real_values = {}

    for col in FEATURE_COLUMNS:
        train_values_col = train_raw[col].values.astype(np.float64)

        cs_transform = ut.CopulaSpaceTransform(time_series=train_values_col)
        train_cs_col = cs_transform.transform()

        mm_transform = ut.MinMaxScaler(time_series=train_cs_col)
        train_mm_cs_col = mm_transform.transform()

        cs_transforms[col] = cs_transform
        mm_transforms[col] = mm_transform
        train_real_values[col] = train_values_col

        print("=" * 80)
        print(col)
        print("train real mean/std:", train_values_col.mean(), train_values_col.std())
        print("train model-space min/max:", train_mm_cs_col.min(), train_mm_cs_col.max())

    return {
        "cs_transforms": cs_transforms,
        "mm_transforms": mm_transforms,
        "train_real_values": train_real_values,
    }


def transform_2ch_real_to_model_space(raw_df: pd.DataFrame, transform_bundle: dict) -> np.ndarray:
    """Transform raw 2ch features to [0,1] model space using train-fitted transforms."""
    cols_model = []

    for col in FEATURE_COLUMNS:
        values = raw_df[col].values.astype(np.float64)
        train_values = transform_bundle["train_real_values"][col]
        mm_transform = transform_bundle["mm_transforms"][col]

        cs = empirical_copula_transform_from_train(values, train_values)
        mm = minmax_transform_from_train(cs, mm_transform, clip=True)
        cols_model.append(mm.astype(np.float32))

    return np.stack(cols_model, axis=-1).astype(np.float32)


def inverse_windows_from_model_space(windows_mm_cs: np.ndarray, transform_bundle: dict) -> np.ndarray:
    """
    Inverse transform windows from model space to real feature space.

    Input shape:  (n_windows, seq_len, 2)
    Output shape: (n_windows, seq_len, 2), channels = FEATURE_COLUMNS
    """
    if windows_mm_cs.ndim != 3:
        raise ValueError("windows_mm_cs must have shape (n_windows, seq_len, n_channels).")
    if windows_mm_cs.shape[2] != len(FEATURE_COLUMNS):
        raise ValueError("Unexpected number of channels.")

    out_cols = []
    for channel_id, col in enumerate(FEATURE_COLUMNS):
        mm_cs_flat = windows_mm_cs[:, :, channel_id].reshape(-1)
        cs_flat = transform_bundle["mm_transforms"][col].inverse_transform(min_max_series=mm_cs_flat)
        real_flat = transform_bundle["cs_transforms"][col].invert_transform(copula_series=cs_flat)
        out_cols.append(real_flat.reshape(windows_mm_cs.shape[0], windows_mm_cs.shape[1]))

    return np.stack(out_cols, axis=-1).astype(np.float32)


In [ ]:

# Diffusion-TS config / train helpers
def recursive_patch_model_shape(obj, seq_len: int, feature_size: int):
    """Recursively patch common shape keys in Diffusion-TS yaml config."""
    if isinstance(obj, dict):
        for key, value in obj.items():
            if key in {"seq_length", "seq_len"}:
                obj[key] = seq_len
            elif key in {"feature_size", "n_feat", "input_size", "enc_in", "c_in", "channels", "var_num"}:
                obj[key] = feature_size
            else:
                recursive_patch_model_shape(value, seq_len, feature_size)
    elif isinstance(obj, list):
        for value in obj:
            recursive_patch_model_shape(value, seq_len, feature_size)


def prepare_2ch_config_for_window(window_id: str) -> tuple[Path, Path]:
    """Create patched 2-channel config and results folder for one WF window."""
    window_root = EXP_ROOT / window_id
    results_dir = window_root / "results"
    window_root.mkdir(parents=True, exist_ok=True)
    results_dir.mkdir(parents=True, exist_ok=True)

    config_path = window_root / f"diff2ch_{INSTRUMENT}_{window_id}_config.yaml"

    if not SOURCE_CONFIG_PATH.exists() and not config_path.exists():
        raise FileNotFoundError(
            f"Neither SOURCE_CONFIG_PATH nor existing config_path exists.\n"
            f"SOURCE_CONFIG_PATH={SOURCE_CONFIG_PATH}\n"
            f"config_path={config_path}"
        )

    base_path = SOURCE_CONFIG_PATH if SOURCE_CONFIG_PATH.exists() else config_path

    with open(base_path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    cfg = copy.deepcopy(cfg)
    recursive_patch_model_shape(cfg.get("model", cfg), SEQ_LEN, INPUT_CHANNELS)

    if "solver" not in cfg:
        raise KeyError("Config must contain a 'solver' section.")

    cfg["solver"]["results_folder"] = str(results_dir)

    with open(config_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

    return config_path, results_dir


class ArgsDiff2Ch:
    def __init__(self, name: str, config_path: Path, save_dir: Path, gpu: int = 0):
        self.name = name
        self.config_path = str(config_path)
        self.save_dir = str(save_dir)
        self.gpu = gpu
        os.makedirs(self.save_dir, exist_ok=True)


def train_conditional_diffusion_2ch(
    train_raw: pd.DataFrame,
    window_id: str,
) -> dict:
    """
    Fit transforms on train_raw, train 2ch Diffusion-TS model, return bundle.
    """
    if len(train_raw) < SEQ_LEN:
        raise ValueError(f"Not enough train liquidation events for {window_id}: {len(train_raw)} < {SEQ_LEN}")

    transform_bundle = fit_2ch_transforms(train_raw)
    train_model_space = transform_2ch_real_to_model_space(train_raw, transform_bundle)

    train_windows = make_windows(
        x=train_model_space,
        seq_len=SEQ_LEN,
        stride=TRAIN_WINDOW_STRIDE,
    )

    print("train_windows shape:", train_windows.shape)

    dataset = NumpyWindowDataset(train_windows)
    train_loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False,
        num_workers=0,
    )

    dl_info = {
        "dataset": dataset,
        "dataloader": train_loader,
        "train_dataloader": train_loader,
        "train_loader": train_loader,
        "test_dataloader": train_loader,
        "test_loader": train_loader,
    }

    config_path, results_dir = prepare_2ch_config_for_window(window_id)
    configs = load_yaml_config(str(config_path))

    args = ArgsDiff2Ch(
        name=f"diff2ch_{INSTRUMENT}_{window_id}",
        config_path=config_path,
        save_dir=results_dir,
        gpu=0,
    )

    device = torch.device(f"cuda:{args.gpu}" if torch.cuda.is_available() else "cpu")
    model = instantiate_from_config(configs["model"]).to(device)

    trainer = Trainer(
        config=configs,
        args=args,
        model=model,
        dataloader=dl_info,
    )

    if LOAD_MILESTONE is not None:
        trainer.load(LOAD_MILESTONE, verbose=True)

    n_params = sum(p.numel() for p in model.parameters())
    print(f"Model params: {n_params / 1e6:.2f}M")
    print("Device:", device)

    if RUN_TRAIN:
        trainer.train()
        print("Training complete.")
    else:
        print("Training skipped.")

    return {
        "trainer": trainer,
        "configs": configs,
        "transform_bundle": transform_bundle,
        "window_id": window_id,
        "train_windows_shape": train_windows.shape,
    }


## 3. Liquidation-event preprocessing

`LIQ_PATH` must contain raw liquidation events. The following columns are required as a minimum:

```text
timestamp
price
amount
```

If the timestamp is already in datetime format, the code will automatically convert it to microseconds.


In [ ]:

# Liquidation-event loader
def _timestamp_to_us(series: pd.Series) -> np.ndarray:
    """Convert timestamp column to int microseconds."""
    if np.issubdtype(series.dtype, np.number):
        return series.astype("int64").to_numpy()

    dt = pd.to_datetime(series, utc=True, errors="coerce")
    return (dt.view("int64") // 1_000).astype("int64")


def load_liquidation_events_2ch(liq_path: Path) -> pd.DataFrame:
    """
    Load all liquidation events and create:
        timestamp_us
        log_delta_t
        log_dollar_amount
    """
    if not liq_path.exists():
        raise FileNotFoundError(f"LIQ_PATH does not exist: {liq_path}")

    df = pd.read_parquet(liq_path)

    required = ["timestamp", "price", "amount"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in {liq_path}: {missing}")

    df = df.copy()
    df["timestamp_us"] = _timestamp_to_us(df["timestamp"])
    df = df.sort_values("timestamp_us").reset_index(drop=True)

    delta_t_sec = pd.Series(df["timestamp_us"]).diff() / 1_000_000.0
    delta_t_sec = delta_t_sec.clip(lower=0.0).fillna(0.0)

    df["log_delta_t"] = np.log1p(delta_t_sec.to_numpy())
    df["log_dollar_amount"] = np.log1p(
        df["price"].astype(float).to_numpy() * df["amount"].astype(float).to_numpy()
    )

    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["timestamp_us"] + FEATURE_COLUMNS).reset_index(drop=True)

    out = df[["timestamp_us"] + FEATURE_COLUMNS].copy()
    out["timestamp_us"] = out["timestamp_us"].astype("int64")

    return out


liq_events_all = load_liquidation_events_2ch(LIQ_PATH)

print("liq_events_all:", liq_events_all.shape)
print("time range us:", liq_events_all["timestamp_us"].min(), "->", liq_events_all["timestamp_us"].max())
display(liq_events_all.head())


liq_events_all: (236067, 3)
time range us: 1761955325274000 -> 1777419146199000


,timestamp_us,log_delta_t,log_dollar_amount
0,1761955325274000,0.000000,9.374207
1,1761955391322000,4.205409,8.092713
2,1761955474046000,4.427526,9.784878
3,1761955514418000,3.722604,5.792556
4,1761955516411000,1.096276,7.089589


## <b>4. Conditional sample -> forecast characteristics</b>

The characteristics are calculated using synthetic future trajectories:

```text
log_delta_t -> delta_t = exp(log_delta_t) - 1
log_dollar_amount -> dollar = exp(log_dollar_amount) - 1
```

For each horizon, we calculate the expected number of liquidations, the expected volume in dollars, p90/p95 and the probability of at least one liquidation.

In [ ]:

def sample_conditional_future_2ch(
    bundle: dict,
    context_real: np.ndarray,
    n_samples: int,
) -> np.ndarray:
    """
    Given one real-space context of shape (HIST_LEN, 2), sample future paths.

    Returns:
        samples_future_real: shape (n_samples, PRED_LEN, 2)
    """
    context_real = np.asarray(context_real, dtype=np.float32)
    if context_real.shape != (HIST_LEN, INPUT_CHANNELS):
        raise ValueError(f"context_real must have shape {(HIST_LEN, INPUT_CHANNELS)}, got {context_real.shape}")

    tmp = pd.DataFrame(context_real, columns=FEATURE_COLUMNS)
    context_model = transform_2ch_real_to_model_space(tmp, bundle["transform_bundle"])

    windows = np.zeros((n_samples, SEQ_LEN, INPUT_CHANNELS), dtype=np.float32)
    windows[:, :HIST_LEN, :] = context_model[None, :, :]

    mask = make_forecast_mask(windows, hist_len=HIST_LEN)

    dataset = ConditionalWindowDataset(windows, mask)
    loader = DataLoader(
        dataset,
        batch_size=SAMPLE_BATCH,
        shuffle=False,
        drop_last=False,
        num_workers=0,
    )

    trainer = bundle["trainer"]

    synthetic, real, masks = trainer.restore(
        raw_dataloader=loader,
        shape=[SEQ_LEN, INPUT_CHANNELS],
        coef=FORECAST_COEF,
        stepsize=FORECAST_STEPSIZE,
        sampling_steps=FORECAST_SAMPLING_STEPS,
    )

    synthetic = to_numpy(synthetic)[:n_samples]

    synthetic_real = inverse_windows_from_model_space(
        synthetic,
        transform_bundle=bundle["transform_bundle"],
    )

    return synthetic_real[:, HIST_LEN:, :]


def summarize_generated_future_paths_2ch(
    samples_future_real: np.ndarray,
    horizons_s=(30, 120, 300),
) -> dict:
    """
    samples_future_real shape: (n_samples, PRED_LEN, 2)
    channels: 0 = log_delta_t, 1 = log_dollar_amount

    Note:
        The horizons are measured from the forecast anchor timestamp.
        The downstream model also receives time_since_diff2ch_anchor, so it can
        learn how stale the forecast is. For production-quality features one can
        additionally build age-adjusted summaries, but this keeps the first test
        simple and cheap.
    """
    log_delta_t = samples_future_real[:, :, 0]
    log_dollar = samples_future_real[:, :, 1]

    delta_t = np.expm1(log_delta_t)
    dollar = np.expm1(log_dollar)

    delta_t = np.maximum(delta_t, 0.0)
    dollar = np.maximum(dollar, 0.0)

    future_time = np.cumsum(delta_t, axis=1)

    out = {}

    for h in horizons_s:
        mask_h = future_time <= h

        n_liq = mask_h.sum(axis=1)
        dollar_sum = (dollar * mask_h).sum(axis=1)

        out[f"diff2ch_mean_n_liq_{h}s"] = float(np.mean(n_liq))
        out[f"diff2ch_p90_n_liq_{h}s"] = float(np.quantile(n_liq, 0.90))
        out[f"diff2ch_prob_any_liq_{h}s"] = float(np.mean(n_liq > 0))

        out[f"diff2ch_mean_dollar_liq_{h}s"] = float(np.mean(dollar_sum))
        out[f"diff2ch_p90_dollar_liq_{h}s"] = float(np.quantile(dollar_sum, 0.90))
        out[f"diff2ch_p95_dollar_liq_{h}s"] = float(np.quantile(dollar_sum, 0.95))

    first_dt = delta_t[:, 0]
    out["diff2ch_mean_time_to_next_liq"] = float(np.mean(first_dt))
    out["diff2ch_p10_time_to_next_liq"] = float(np.quantile(first_dt, 0.10))
    out["diff2ch_prob_liq_within_5s"] = float(np.mean(first_dt <= 5.0))
    out["diff2ch_prob_liq_within_30s"] = float(np.mean(first_dt <= 30.0))

    return out


def make_forecast_anchor_indices(
    timestamps: np.ndarray,
    context_len: int,
    stride_events: int,
    max_anchors: int | None = None,
    anchor_start_ts: int | None = None,
    anchor_end_ts: int | None = None,
    selection: str = "uniform",
) -> np.ndarray:
    """
    Build forecast anchor indices.

    Critical fix:
        If max_anchors is set, do NOT take the first max_anchors anchors.
        That covers only the beginning of the period and makes all test features
        constant after join_asof. Instead, select anchors uniformly over the
        whole requested time range.

    Parameters
    ----------
    timestamps:
        All liquidation timestamps in microseconds.
    context_len:
        Number of known historical events used as prefix.
    stride_events:
        Candidate anchor stride in event index units.
    max_anchors:
        Optional cap for a quick smoke test.
    anchor_start_ts, anchor_end_ts:
        Forecast anchors are restricted to this timestamp interval.
        Context is still allowed to use earlier liquidation events.
    selection:
        "uniform" or "first". Use "uniform" for any capped run.
    """
    timestamps = np.asarray(timestamps, dtype=np.int64)

    if len(timestamps) <= context_len:
        return np.array([], dtype=np.int64)

    candidate_indices = np.arange(
        context_len,
        len(timestamps),
        stride_events,
        dtype=np.int64,
    )

    # Anchor timestamp is the last known event inside the historical context.
    candidate_anchor_ts = timestamps[candidate_indices - 1]

    keep = np.ones(len(candidate_indices), dtype=bool)

    if anchor_start_ts is not None:
        keep &= candidate_anchor_ts >= int(anchor_start_ts)

    if anchor_end_ts is not None:
        keep &= candidate_anchor_ts <= int(anchor_end_ts)

    candidate_indices = candidate_indices[keep]

    if len(candidate_indices) == 0:
        return candidate_indices

    if max_anchors is not None and len(candidate_indices) > max_anchors:
        if selection == "uniform":
            pick = np.linspace(
                0,
                len(candidate_indices) - 1,
                int(max_anchors),
            ).round().astype(np.int64)
            candidate_indices = candidate_indices[pick]
        elif selection == "first":
            candidate_indices = candidate_indices[:int(max_anchors)]
        else:
            raise ValueError("selection must be either 'uniform' or 'first'.")

    return candidate_indices.astype(np.int64)


def build_diff2ch_forecast_table_for_window(
    liq_events: pd.DataFrame,
    bundle: dict,
    stride_events: int = 16,
    max_anchors: int | None = None,
    anchor_start_ts: int | None = None,
    anchor_end_ts: int | None = None,
    anchor_selection: str = "uniform",
) -> pd.DataFrame:
    """
    Build forecast table on liquidation-event anchors.

    Output columns:
        diff2ch_anchor_ts
        diff2ch_anchor_event_idx
        diff2ch_*

    Important:
        With max_anchors set, anchors are selected uniformly over
        [anchor_start_ts, anchor_end_ts]. This prevents constant test features.
    """
    values = liq_events[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
    timestamps = liq_events["timestamp_us"].to_numpy(dtype=np.int64)

    if len(liq_events) <= HIST_LEN:
        raise ValueError(f"Not enough liq events to build contexts: {len(liq_events)} <= {HIST_LEN}")

    anchor_indices = make_forecast_anchor_indices(
        timestamps=timestamps,
        context_len=HIST_LEN,
        stride_events=stride_events,
        max_anchors=max_anchors,
        anchor_start_ts=anchor_start_ts,
        anchor_end_ts=anchor_end_ts,
        selection=anchor_selection,
    )

    if len(anchor_indices) == 0:
        raise ValueError(
            "No forecast anchors were selected. Check anchor_start_ts/anchor_end_ts, "
            "HIST_LEN and liquidation-event coverage."
        )

    anchor_ts = timestamps[anchor_indices - 1]
    print("forecast anchor candidates selected:", len(anchor_indices))
    print("forecast anchor ts range:", int(anchor_ts.min()), "->", int(anchor_ts.max()))
    print("forecast anchor event idx range:", int(anchor_indices.min() - 1), "->", int(anchor_indices.max() - 1))

    rows = []

    for idx in tqdm(anchor_indices, desc="diff2ch forecast anchors"):
        context = values[idx - HIST_LEN:idx]

        samples_future_real = sample_conditional_future_2ch(
            bundle=bundle,
            context_real=context,
            n_samples=N_SAMPLES,
        )

        feats = summarize_generated_future_paths_2ch(
            samples_future_real=samples_future_real,
            horizons_s=HORIZONS_S,
        )

        feats["diff2ch_anchor_ts"] = int(timestamps[idx - 1])
        feats["diff2ch_anchor_event_idx"] = int(idx - 1)

        rows.append(feats)

    out = pd.DataFrame(rows)

    # Keep aux columns as int64; forecast values as float32.
    out["diff2ch_anchor_ts"] = out["diff2ch_anchor_ts"].astype("int64")
    out["diff2ch_anchor_event_idx"] = out["diff2ch_anchor_event_idx"].astype("int64")

    value_cols = [c for c in out.columns if c.startswith("diff2ch_") and c not in {"diff2ch_anchor_ts", "diff2ch_anchor_event_idx"}]
    for c in value_cols:
        out[c] = out[c].astype("float32")

    return out


## <b>5. Join diffusion features to split data</b>

`diff2ch_*` features are joined via `join_asof` using the latest available forecast anchor.

The following are also calculated:

```text
time_since_diff2ch_anchor
time_since_last_liq
```

`time_since_last_liq` is calculated based on all actual liquidation timestamps, not just sparse forecast anchors.

In [8]:
# ============================================================
# Join helpers
# ============================================================
DIFF2CH_AUX_COLUMNS = [
    "diff2ch_anchor_ts",
    "diff2ch_anchor_event_idx",
]


def get_diff2ch_forecast_value_columns() -> list[str]:
    """Only model-value forecast columns, without aux anchor columns."""
    cols = []
    for h in HORIZONS_S:
        cols.extend([
            f"diff2ch_mean_n_liq_{h}s",
            f"diff2ch_p90_n_liq_{h}s",
            f"diff2ch_prob_any_liq_{h}s",
            f"diff2ch_mean_dollar_liq_{h}s",
            f"diff2ch_p90_dollar_liq_{h}s",
            f"diff2ch_p95_dollar_liq_{h}s",
        ])

    cols.extend([
        "diff2ch_mean_time_to_next_liq",
        "diff2ch_p10_time_to_next_liq",
        "diff2ch_prob_liq_within_5s",
        "diff2ch_prob_liq_within_30s",
    ])

    return cols


DIFF2CH_FORECAST_VALUE_COLUMNS = get_diff2ch_forecast_value_columns()


def add_diff2ch_features_to_split_df(
    split_df: pl.DataFrame,
    forecast_df: pd.DataFrame,
    liq_events: pd.DataFrame,
) -> pl.DataFrame:
    """
    Join forecast features and last-liq timestamp to a split dataframe.

    Patch details:
      - Forecast value columns are filled/cast separately.
      - Aux columns diff2ch_anchor_ts/diff2ch_anchor_event_idx stay Int64.
      - time_since_* columns are Float32 model features.
      - We add diff2ch_has_anchor so zero-filled early rows are identifiable.
    """
    forecast_pl = pl.from_pandas(forecast_df).sort("diff2ch_anchor_ts")

    # Enforce stable dtypes before join.
    forecast_pl = forecast_pl.with_columns([
        pl.col("diff2ch_anchor_ts").cast(pl.Int64),
        pl.col("diff2ch_anchor_event_idx").cast(pl.Int64),
    ] + [
        pl.col(c).cast(pl.Float32)
        for c in DIFF2CH_FORECAST_VALUE_COLUMNS
        if c in forecast_pl.columns
    ])

    last_liq_pl = pl.DataFrame({
        "last_liq_timestamp": liq_events["timestamp_us"].to_numpy(dtype=np.int64),
    }).sort("last_liq_timestamp")

    out = split_df.sort("timestamp")

    out = out.join_asof(
        forecast_pl,
        left_on="timestamp",
        right_on="diff2ch_anchor_ts",
        strategy="backward",
    )

    out = out.join_asof(
        last_liq_pl,
        left_on="timestamp",
        right_on="last_liq_timestamp",
        strategy="backward",
    )

    out = out.with_columns([
        pl.col("diff2ch_anchor_ts").is_not_null().cast(pl.Float32).alias("diff2ch_has_anchor"),

        ((pl.col("timestamp") - pl.col("diff2ch_anchor_ts")) / 1_000_000.0)
        .cast(pl.Float32)
        .alias("time_since_diff2ch_anchor"),

        ((pl.col("timestamp") - pl.col("last_liq_timestamp")) / 1_000_000.0)
        .cast(pl.Float32)
        .alias("time_since_last_liq"),
    ])

    # Fill only real forecast value features, not aux anchor timestamps.
    out = out.with_columns(
        [pl.col(c).fill_null(0.0).cast(pl.Float32) for c in DIFF2CH_FORECAST_VALUE_COLUMNS]
        + [
            pl.col("diff2ch_has_anchor").fill_null(0.0).cast(pl.Float32),
            pl.col("time_since_diff2ch_anchor").fill_null(999999.0).cast(pl.Float32),
            pl.col("time_since_last_liq").fill_null(999999.0).cast(pl.Float32),
        ]
    )

    return out


def get_diff2ch_feature_list() -> list[str]:
    # These are the columns that should be passed into the PnL model.
    # Aux columns diff2ch_anchor_ts / diff2ch_anchor_event_idx are debug columns,
    # not model features.
    return (
        DIFF2CH_FORECAST_VALUE_COLUMNS
        + [
            "diff2ch_has_anchor",
            "time_since_diff2ch_anchor",
            "time_since_last_liq",
        ]
    )


DIFFUSION_2CH_FEATURES = get_diff2ch_feature_list()
print("n DIFFUSION_2CH_FEATURES:", len(DIFFUSION_2CH_FEATURES))
print(DIFFUSION_2CH_FEATURES)


n DIFFUSION_2CH_FEATURES: 25
['diff2ch_mean_n_liq_30s', 'diff2ch_p90_n_liq_30s', 'diff2ch_prob_any_liq_30s', 'diff2ch_mean_dollar_liq_30s', 'diff2ch_p90_dollar_liq_30s', 'diff2ch_p95_dollar_liq_30s', 'diff2ch_mean_n_liq_120s', 'diff2ch_p90_n_liq_120s', 'diff2ch_prob_any_liq_120s', 'diff2ch_mean_dollar_liq_120s', 'diff2ch_p90_dollar_liq_120s', 'diff2ch_p95_dollar_liq_120s', 'diff2ch_mean_n_liq_300s', 'diff2ch_p90_n_liq_300s', 'diff2ch_prob_any_liq_300s', 'diff2ch_mean_dollar_liq_300s', 'diff2ch_p90_dollar_liq_300s', 'diff2ch_p95_dollar_liq_300s', 'diff2ch_mean_time_to_next_liq', 'diff2ch_p10_time_to_next_liq', 'diff2ch_prob_liq_within_5s', 'diff2ch_prob_liq_within_30s', 'diff2ch_has_anchor', 'time_since_diff2ch_anchor', 'time_since_last_liq']


## <b>6. Main loop: create `*_splits_diff2ch` Parquet files</b>

For each window:

1. load `train_model/calib/test`;
2. take the train liquidation events for the `train_model` period;
3. train the 2ch conditional diffusion model using only the train events;
4. construct a forecast table based on liquidation anchors within the full window range;
5. join the `diff2ch_*` files to `train_model/calib/test`;
6. save the new Parquet files.


In [ ]:
# Main generation loop
summary_rows = []

for wid in WINDOWS_TO_RUN:
    print("\n" + "=" * 100)
    print(f"WINDOW {wid}")
    print("=" * 100)

    out_paths = {
        "train_model": OUT_SPLIT_DIR / f"{INSTRUMENT}_{wid}_train_model.parquet",
        "calib": OUT_SPLIT_DIR / f"{INSTRUMENT}_{wid}_calib.parquet",
        "test": OUT_SPLIT_DIR / f"{INSTRUMENT}_{wid}_test.parquet",
    }

    if (not OVERWRITE) and all(p.exists() for p in out_paths.values()):
        print(f"{wid}: output files already exist, skip.")
        continue

    split = load_split_dataset(
        split_dir=BASE_SPLIT_DIR,
        instrument=INSTRUMENT,
        window_id=wid,
    )

    t_train_min = int(split.train_model["timestamp"].min())
    t_train_max = int(split.train_model["timestamp"].max())

    t_all_min = int(min(
        split.train_model["timestamp"].min(),
        split.calib["timestamp"].min(),
        split.test["timestamp"].min(),
    ))
    t_all_max = int(max(
        split.train_model["timestamp"].max(),
        split.calib["timestamp"].max(),
        split.test["timestamp"].max(),
    ))

    print("train_model rows:", split.train_model.height)
    print("calib rows:      ", split.calib.height)
    print("test rows:       ", split.test.height)
    print("train time range:", t_train_min, "->", t_train_max)
    print("all time range:  ", t_all_min, "->", t_all_max)

    liq_train = liq_events_all[
        (liq_events_all["timestamp_us"] >= t_train_min)
        & (liq_events_all["timestamp_us"] <= t_train_max)
    ].copy().reset_index(drop=True)

    # Use all historical liquidation events up to window end.
    # Forecast anchors are restricted to [t_all_min, t_all_max],
    # while each anchor context is allowed to use earlier events.
    # This does not leak future into a given row because features are joined backward.
    liq_all_until_window_end = liq_events_all[
        liq_events_all["timestamp_us"] <= t_all_max
    ].copy().reset_index(drop=True)

    print("liq_train events:", len(liq_train))
    print("liq_all_until_window_end events:", len(liq_all_until_window_end))

    if len(liq_train) < SEQ_LEN:
        raise ValueError(f"{wid}: too few train liquidation events: {len(liq_train)} < {SEQ_LEN}")
    if len(liq_all_until_window_end) < HIST_LEN:
        raise ValueError(f"{wid}: too few total liquidation events for contexts.")

    # 1. Train diffusion on train_model liquidation events only.
    bundle = train_conditional_diffusion_2ch(
        train_raw=liq_train,
        window_id=wid,
    )

    # 2. Build forecast table.
    # Critical: anchors are spread uniformly across the whole split period
    # when MAX_FORECAST_ANCHORS_PER_WINDOW is set.
    forecast_df = build_diff2ch_forecast_table_for_window(
        liq_events=liq_all_until_window_end,
        bundle=bundle,
        stride_events=FORECAST_ANCHOR_STRIDE_EVENTS,
        max_anchors=MAX_FORECAST_ANCHORS_PER_WINDOW,
        anchor_start_ts=t_all_min,
        anchor_end_ts=t_all_max,
        anchor_selection="uniform",
    )

    print("forecast_df shape:", forecast_df.shape)
    print("forecast_df anchor range:", forecast_df["diff2ch_anchor_ts"].min(), "->", forecast_df["diff2ch_anchor_ts"].max())
    display(forecast_df.head())

    # 3. Join features to split parts.
    train_model_out = add_diff2ch_features_to_split_df(
        split_df=split.train_model,
        forecast_df=forecast_df,
        liq_events=liq_all_until_window_end,
    )
    calib_out = add_diff2ch_features_to_split_df(
        split_df=split.calib,
        forecast_df=forecast_df,
        liq_events=liq_all_until_window_end,
    )
    test_out = add_diff2ch_features_to_split_df(
        split_df=split.test,
        forecast_df=forecast_df,
        liq_events=liq_all_until_window_end,
    )

    missing = [c for c in DIFFUSION_2CH_FEATURES if c not in train_model_out.columns]
    if missing:
        raise ValueError(f"{wid}: missing diffusion columns after join: {missing}")

    parts = {
        "train_model": train_model_out,
        "calib": calib_out,
        "test": test_out,
    }

    # 4. Diagnostics before saving.
    for part_name, df_part in parts.items():
        null_total = int(df_part.select(DIFFUSION_2CH_FEATURES).null_count().sum_horizontal()[0])
        n_unique = df_part.select([
            pl.col(c).n_unique().alias(c)
            for c in DIFFUSION_2CH_FEATURES
        ])
        constant_cols = [
            c for c in DIFFUSION_2CH_FEATURES
            if int(n_unique[c][0]) <= 1
        ]

        anchor_diag = df_part.select([
            pl.col("timestamp").min().alias("ts_min"),
            pl.col("timestamp").max().alias("ts_max"),
            pl.col("diff2ch_anchor_ts").min().alias("anchor_ts_min"),
            pl.col("diff2ch_anchor_ts").max().alias("anchor_ts_max"),
            pl.col("diff2ch_anchor_ts").n_unique().alias("n_unique_anchor_ts"),
            pl.col("time_since_diff2ch_anchor").median().alias("age_anchor_median"),
            pl.col("time_since_diff2ch_anchor").quantile(0.95).alias("age_anchor_p95"),
            pl.col("time_since_last_liq").median().alias("age_last_liq_median"),
            pl.col("time_since_last_liq").quantile(0.95).alias("age_last_liq_p95"),
        ])

        print(f"\n{wid} {part_name}:")
        print(f"  rows={df_part.height:,}, cols={len(df_part.columns)}, diffusion_nulls={null_total}")
        print(f"  constant model diffusion cols: {len(constant_cols)}")
        if constant_cols:
            print(f"  first constant cols: {constant_cols[:10]}")
        display(anchor_diag)

    # 5. Save.
    for part_name, df_part in parts.items():
        out_path = out_paths[part_name]
        df_part.write_parquet(out_path, compression="zstd", statistics=True)
        print(f"saved {out_path.name}: {df_part.height:,} rows, {len(df_part.columns)} cols")

    summary_rows.append({
        "window": wid,
        "train_rows": train_model_out.height,
        "calib_rows": calib_out.height,
        "test_rows": test_out.height,
        "liq_train_events": len(liq_train),
        "forecast_rows": len(forecast_df),
        "forecast_anchor_min": int(forecast_df["diff2ch_anchor_ts"].min()),
        "forecast_anchor_max": int(forecast_df["diff2ch_anchor_ts"].max()),
        "out_dir": str(OUT_SPLIT_DIR),
    })

    del split, train_model_out, calib_out, test_out, liq_train, liq_all_until_window_end, forecast_df, bundle
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

summary_df = pd.DataFrame(summary_rows)
display(summary_df)


## <b>7. Validate output split files</b>

In [10]:
# ============================================================
# Validate output split files
# ============================================================
for wid in WINDOWS_TO_RUN:
    print("\n" + "=" * 80)
    print(wid)

    split = load_split_dataset(
        split_dir=OUT_SPLIT_DIR,
        instrument=INSTRUMENT,
        window_id=wid,
    )

    for part_name, df_part in [
        ("train_model", split.train_model),
        ("calib", split.calib),
        ("test", split.test),
    ]:
        missing = [c for c in DIFFUSION_2CH_FEATURES if c not in df_part.columns]
        if missing:
            raise ValueError(f"{wid} {part_name}: missing {missing}")

        null_total = int(df_part.select(DIFFUSION_2CH_FEATURES).null_count().sum_horizontal()[0])

        n_unique_df = df_part.select([
            pl.col(c).n_unique().alias(c)
            for c in DIFFUSION_2CH_FEATURES
        ])

        constant_cols = [
            c for c in DIFFUSION_2CH_FEATURES
            if int(n_unique_df[c][0]) <= 1
        ]

        print(
            f"{part_name:12s}: rows={df_part.height:,}, "
            f"cols={len(df_part.columns)}, diffusion_nulls={null_total}, "
            f"constant_model_cols={len(constant_cols)}"
        )

        if constant_cols:
            print("  WARNING constant cols:", constant_cols[:20])

    del split
    gc.collect()



W1
train_model : rows=3,285,262, cols=53, diffusion_nulls=0, constant_model_cols=0
calib       : rows=424,282, cols=53, diffusion_nulls=0, constant_model_cols=1
  WARNING constant cols: ['diff2ch_has_anchor']
test        : rows=1,971,880, cols=53, diffusion_nulls=0, constant_model_cols=1
  WARNING constant cols: ['diff2ch_has_anchor']

W2
train_model : rows=5,739,389, cols=53, diffusion_nulls=0, constant_model_cols=0
calib       : rows=720,356, cols=53, diffusion_nulls=0, constant_model_cols=1
  WARNING constant cols: ['diff2ch_has_anchor']
test        : rows=2,068,316, cols=53, diffusion_nulls=0, constant_model_cols=1
  WARNING constant cols: ['diff2ch_has_anchor']

W3
train_model : rows=7,275,187, cols=53, diffusion_nulls=0, constant_model_cols=0
calib       : rows=1,719,093, cols=53, diffusion_nulls=0, constant_model_cols=1
  WARNING constant cols: ['diff2ch_has_anchor']
test        : rows=2,766,251, cols=53, diffusion_nulls=0, constant_model_cols=1
  WARNING constant cols: ['diff2

In [ ]:
# Copy this cell to notebook.

DIFFUSION_2CH_FEATURES = [
    "diff2ch_mean_n_liq_30s",
    "diff2ch_p90_n_liq_30s",
    "diff2ch_prob_any_liq_30s",
    "diff2ch_mean_dollar_liq_30s",
    "diff2ch_p90_dollar_liq_30s",
    "diff2ch_p95_dollar_liq_30s",

    "diff2ch_mean_n_liq_120s",
    "diff2ch_p90_n_liq_120s",
    "diff2ch_prob_any_liq_120s",
    "diff2ch_mean_dollar_liq_120s",
    "diff2ch_p90_dollar_liq_120s",
    "diff2ch_p95_dollar_liq_120s",

    "diff2ch_mean_n_liq_300s",
    "diff2ch_p90_n_liq_300s",
    "diff2ch_prob_any_liq_300s",
    "diff2ch_mean_dollar_liq_300s",
    "diff2ch_p90_dollar_liq_300s",
    "diff2ch_p95_dollar_liq_300s",

    "diff2ch_mean_time_to_next_liq",
    "diff2ch_p10_time_to_next_liq",
    "diff2ch_prob_liq_within_5s",
    "diff2ch_prob_liq_within_30s",

    "diff2ch_has_anchor",
    "time_since_diff2ch_anchor",
    "time_since_last_liq",
]